# Lab: 2D Convolution on the Raspberry Pi
## Python → C → ARM64 Assembly

In this lab you run the **same operation**, a 2D convolution, written in three languages on
a Raspberry Pi 5, and then step by step in assembly. You will measure how long each version
takes and find out *why* the lower levels are faster.

You will not write a program from scratch. The code is ready and commented. Your job is to
**take a photo with the Raspberry Pi camera, run the code on it, read it, change one piece
of assembly, and explain what you see** with your own numbers.

The camera must be connected **before** the Raspberry Pi is powered on.

### How to use this notebook

- Run a cell with **Shift + Enter**. Run the cells **in order**, from top to bottom.
- A cell marked **✏️** is a question: double-click it, write your answer below
  **Your answer:**, then Shift + Enter.
- Save often (**Ctrl + S**).

### What you hand in

This notebook, **exported as HTML** with all outputs and answers (see the last section).

---
## Part 0. Check the setup

The cell below checks that the system is 64-bit ARM (`aarch64`), that the camera is
detected, builds the C and assembly code with `make`, and runs a correctness test: every
version must give exactly the same result as Python on 13 image sizes.

**Expected:** `aarch64`, a camera in the list (for example `0 : ov5647 [2592x1944 ...]`),
then the build, then a line ending in `all OK`.

If it says `No cameras available!`: shut the Pi down, unplug it, check the camera cable at
both ends, and power it on again. The camera is only detected if it is connected **before**
the Pi is powered on.

In [ ]:
!uname -m
!cat /proc/device-tree/model; echo
!rpicam-hello --list-cameras 2>/dev/null
!make
!python3 conv_bench.py --test

---
## Part 1. What is a convolution?

A **kernel** is a small grid of numbers, here 3 × 3. It slides over the image. At each
position, every pixel under the kernel is multiplied by the kernel number on top of it, and
the 9 products are added. That sum is one pixel of the output:

$$
\text{out}[y][x] = \sum_{k_y=0}^{2}\ \sum_{k_x=0}^{2}\ \text{in}[y+k_y][x+k_x] \cdot k[k_y][k_x]
$$

The kernel never hangs outside the image, so the output is 2 pixels smaller in each
direction. Image filters (blur, edge detection) and the neural networks that recognize
images are built on exactly this operation.

Run the cell below. It shows a small 6 × 8 image, the kernel, and how output pixel
**(0, 0)** is computed: the yellow 3 × 3 window is multiplied by the kernel and summed.

In [ ]:
from conv_demo import notebook_image, notebook_race, notebook_step, take_photo

notebook_step(0, 0)

✏️ **Q1.** Compute output pixel **out[1][2] by hand** for this new image and kernel. Write the
9 products and the result. *(Rows and columns are numbered from 0. The window for
out[1][2] starts at row 1, column 2 of the input.)*

**Input image** (5 × 6)

| | col 0 | col 1 | col 2 | col 3 | col 4 | col 5 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| **row 0** | 8 | 6 | 2 | 2 | 6 | 6 |
| **row 1** | 5 | 8 | 9 | 6 | 5 | 2 |
| **row 2** | 4 | 7 | 7 | 9 | 5 | 3 |
| **row 3** | 0 | 7 | 3 | 1 | 7 | 9 |
| **row 4** | 4 | 8 | 9 | 6 | 0 | 1 |

**Kernel** (3 × 3)

| | col 0 | col 1 | col 2 |
| --- | ---: | ---: | ---: |
| **row 0** | 0 | −2 | 1 |
| **row 1** | 1 | 1 | −1 |
| **row 2** | −1 | −2 | 2 |

**Your answer:** *(double-click this cell and write it here)*

### The image of this lab: your photo

Every part of this lab processes **a photo you take now with the Raspberry Pi camera**.
To a computer a grayscale photo is just a grid of numbers, one per pixel, from 0 (black) to
255 (white). Whatever the photo shows, every version does exactly the same amount of work,
so everybody's times are comparable.

The lab uses the **edge-detection kernel**

```
-1 -1 -1
-1  8 -1
-1 -1 -1
```

multiplies the centre pixel by 8 and subtracts its 8 neighbours. Where the brightness does
not change (a wall, the sky) the result is 0; where it changes (the outline of a face, a
letter), the result is large.

Run the cell below. A camera preview opens for 3 seconds, then the Raspberry Pi takes a
grayscale photo of **2592 × 1944 pixels** and saves it as `photo.pgm`. It does not have to
be your face: point the camera at anything with clear shapes (a hand, a book, a drawing).
The photo will be part of the notebook you hand in.

If the cell says **No camera detected**, follow the steps it gives and run it again. To
retake the photo, just run the cell again: the new photo replaces the old one.

In [ ]:
take_photo()
notebook_image()

✏️ **Q2.** Your photo is 2592 × 1944 pixels. How many output pixels does the convolution
produce, and how many **multiply-adds** does it need for the whole photo? Show the
calculation.

**Your answer:** *(double-click this cell and write it here)*

✏️ **Q3.** In the output, which parts of your photo turned dark (strong edges) and which
stayed white? Use the kernel to explain why a flat area gives 0. *(Hint: add up the 9
numbers of the kernel.)*

**Your answer:** *(double-click this cell and write it here)*

---
## Part 2. The same loops in three languages

### Python

Four nested loops: every output pixel, then every kernel position.

```python
for y in range(oh):                  # every output row ...
    for x in range(ow):              # ... every output column
        acc = 0
        for ky in range(3):          # every kernel row
            for kx in range(3):      # every kernel column
                acc += img[y + ky][x + kx] * k[ky][kx]
        out[y][x] = acc
```

Python does not run this directly on the CPU. The **interpreter** reads each operation
while the program runs: it looks up `img`, finds row `y + ky` in a list, finds the element,
checks that both values are numbers, creates a new integer object for the product, and so
on. Each `+` and `*` costs many steps.

### C (`src/conv.c`): two versions

The same four loops. The difference is *when* the work happens: the compiler (`gcc -O2`)
translates them to machine instructions **once**, before the program runs.

The C file has **two versions that differ only in the order of the two outer loops**:

```c
// C v1: column order                         // C v2: row order
for (long x = 0; x < ow; x++)                 for (long y = 0; y < oh; y++)
    for (long y = 0; y < oh; y++) {               for (long x = 0; x < ow; x++) {
        int32_t acc = 0;                              int32_t acc = 0;
        for (long ky = 0; ky < 3; ky++)               for (long ky = 0; ky < 3; ky++)
          for (long kx = 0; kx < 3; kx++)               for (long kx = 0; kx < 3; kx++)
            acc += in[(y+ky)*w + (x+kx)]                  acc += in[(y+ky)*w + (x+kx)]
                   * k[ky*3 + kx];                               * k[ky*3 + kx];
        out[y * ow + x] = acc;                        out[y * ow + x] = acc;
    }                                             }
```

Both compute exactly the same image. To see why their speed can differ, you need to know
how the image sits in memory:

- The image is one flat block of memory, **row after row**, so pixel `[r][c]` is at index
  `r * w + c`. Neighbours in a row are neighbours in memory; neighbours in a column are a
  whole row apart (2592 × 4 bytes = 10,368 bytes).
- The CPU never reads one pixel from RAM. It brings a whole **cache line** of 64 bytes
  (16 pixels) into its fast **cache** memory, expecting the next pixels to be used soon.

**C v2** walks along the rows, so each cache line is used for 16 pixels in a row. **C v1**
walks down the columns: every step jumps 10,368 bytes, needs a different cache line, and the
cache has to keep thousands of lines to reuse them for the next column.

The compiler does **not** fix the order for you: it is the programmer's choice.

### How Python calls C and assembly

`make` turns `src/conv.c` and `src/conv_asm.S` into **shared libraries** (`build/*.so`).
`conv_bench.py` loads them with Python's `ctypes` module and calls the functions as if they
were Python functions, passing the image as a block of memory. Python only measures the
time and checks the output; the work runs as native machine code.

### From RISC-V (Ripes) to ARM64

In Ripes you wrote **RISC-V** assembly. The Raspberry Pi's CPU speaks **ARM64**
(also called AArch64). Both are RISC architectures built on the same ideas: registers,
instructions with a destination and two operands, and memory reached only through loads
and stores. What changes is mostly how things are written. Use these two tables as a
dictionary while you read the ARM64 code.

**Registers**

| Use | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| Function arguments and result | `a0`–`a7` | `x0`–`x7` |
| Temporaries (free to overwrite) | `t0`–`t6` | `x9`–`x15` (also `x16`, `x17`) |
| Saved (must be restored before returning) | `s0`–`s11` | `x19`–`x28` |
| Return address | `ra` | `x30` (also called `lr`) |
| Stack pointer | `sp` | `sp` |
| Always zero | `zero` | `xzr` / `wzr` |
| Size | 32 bits (RV32) | `xN` = 64 bits; `wN` = the low 32 bits of `xN` |

In this lab, pixels are 32-bit numbers, so pixel arithmetic uses `w` registers; memory
addresses and loop counters use `x` registers.

**Instructions**

| What it does | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| Load a constant | `li t0, 3` | `mov x15, #3` |
| Copy a register | `mv s4, s2` | `mov w4, w2` |
| Add / add a constant | `add t0, t1, t2` / `addi t0, t0, 1` | `add x0, x1, x2` / `add x0, x0, #1` |
| Multiply | `mul t0, t1, t2` | `mul w0, w1, w2` |
| Shift left | `slli t0, t0, 2` | `lsl x0, x0, #2` |
| Load a word from memory | `lw t0, 4(a0)` | `ldr w0, [x0, #4]` |
| Store a word | `sw t0, 0(a1)` | `str w0, [x1]` |
| Load, then advance the pointer | `lw t0, 0(a0)` + `addi a0, a0, 4` | `ldr w0, [x0], #4` (one instruction) |
| Load `array[index]` (4-byte words) | `slli` + `add` + `lw` | `ldr w0, [x1, x2, lsl #2]` (one instruction) |
| Branch if less than | `blt t0, t1, loop` | `cmp x0, x1` then `b.lt loop` |
| Branch if (not) zero | `beqz t0, done` / `bnez t0, loop` | `cbz x0, done` / `cbnz x0, loop` |
| Count down and loop | `addi t0, t0, -1` + `bnez t0, loop` | `subs x0, x0, #1` then `b.ne loop` |
| Jump / return | `j loop` / `ret` | `b loop` / `ret` |
| Multiply and add (MAC) | `mul` + `add` (RV32IM has no MAC) | `madd w0, w1, w2, w0` |
| Comment | `# ...` | `// ...` |

Three things to remember: in ARM64 constants start with `#`, memory operands go inside
`[ ]`, and comparisons are separate from branches (`cmp` sets flags, `b.lt` / `b.ne` read
them).

### Assembly v1: brute force (`src/conv_asm.S`)

The first assembly version is a direct translation of the C loops. Below is its innermost
loop, the one that runs once per multiply-add: on the left, as you would write it in Ripes;
on the right, the real ARM64 code that runs on the Pi. Registers:
`a0`/`x0` = `in`, `a2`/`x2` = `k`, `a4`/`x4` = `w`, `s0`/`x7` = `x`, `s1`/`x8` = `y`,
`s2`/`x10` = `ky`, `s3`/`x11` = `kx`, `s4`/`w9` = `acc`.

```
RISC-V (Ripes)                            ARM64 (Raspberry Pi)
inner:                                    4:
  add  t0, s1, s2     # y + ky              add  x12, x8, x10            // y + ky
  mul  t0, t0, a4     # (y + ky) * w        mul  x12, x12, x4            // (y + ky) * w
  add  t1, s0, s3     # x + kx              add  x13, x7, x11            // x + kx
  add  t0, t0, t1     # index into in[]     add  x12, x12, x13           // index into in[]
  slli t0, t0, 2      # index * 4 bytes
  add  t0, a0, t0     # address of pixel
  lw   t2, 0(t0)      # pixel = in[index]   ldr  w14, [x0, x12, lsl #2]  // pixel = in[index]
  li   t3, 3                                mov  x15, #3
  mul  t3, s2, t3     # ky * 3              mul  x15, x10, x15           // ky * 3
  add  t3, t3, s3     # index into k[]      add  x15, x15, x11           // index into k[]
  slli t3, t3, 2      # index * 4 bytes
  add  t3, a2, t3     # address of coef
  lw   t4, 0(t3)      # coef = k[index]     ldr  w16, [x2, x15, lsl #2]  // coef = k[index]
  mul  t2, t2, t4     # pixel * coef        mul  w14, w14, w16           // pixel * coef
  add  s4, s4, t2     # acc += pixel*coef   add  w9, w9, w14             // acc += pixel * coef
  addi s3, s3, 1      # kx++                add  x11, x11, #1            // kx++
  li   t5, 3                                cmp  x11, #3
  blt  s3, t5, inner  # while kx < 3        b.lt 4b                      // repeat while kx < 3
```

`ldr w14, [x0, x12, lsl #2]` does in one instruction what takes three in RISC-V: it
multiplies the index by 4 (`lsl #2`, because each pixel is 4 bytes), adds it to the address
of the image, and loads the pixel. `4b` means "label `4`, backwards".

✏️ **Q4.** In the ARM64 assembly v1 loop above (right column):
1. How many instructions run for **one** multiply-add?
2. How many of them do the useful work (load the pixel, load the coefficient, multiply,
   add to `acc`), and how many only compute indexes or control the loop?
3. The RISC-V version (left column) needs 18 instructions for the same work. Which ARM64
   instructions do the job of several RISC-V ones?

**Your answer:** *(double-click this cell and write it here)*

---
## Part 3. Measure all the versions

The cell below runs every version on your photo, with the edge-detection kernel. Python
runs once (**about 9 seconds**); each C and assembly version runs 5 times and the median is
shown. Every output is compared with Python's: `OK` means identical.

Close other programs before running it.

In [ ]:
!python3 conv_bench.py

✏️ **Q5.** Copy your numbers into the table (two decimals).

| Version | Time (ms) | vs Python |
| --- | ---: | ---: |
| Python | | 1.0x |
| C v1 column order | | |
| C v2 row order | | |
| ASM v1 brute force | | |
| ASM v2 pointers | | |
| ASM v3 MAC (yours) | | |
| ASM v4 NEON | | |

*(Double-click to fill it in.)*

✏️ **Q6.** Where was the **biggest** jump in speed, and why?
Is a direct translation to assembly (ASM v1) automatically faster than C v2? Look at
your numbers.

**Your answer:** *(double-click this cell and write it here)*

✏️ **Q7.** C v1 and C v2 are the same language, the same compiler and the same
operations; only two lines are swapped. How many times faster is C v2 than C v1 on your
Pi? Explain the difference with the words *memory*, *row* and *cache line*.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 4. Climbing the assembly ladder

Each assembly version changes **one idea** compared to the previous one, so each speed-up
has one cause. Open `src/conv_asm.S` to read the full code; every line is commented.

### v1 → v2: stop computing indexes inside the loop

v1 recomputes `(y + ky) * w + (x + kx)` with multiplications for every single multiply-add,
and reloads the kernel from memory every time. v2 instead:

- loads the 9 kernel numbers **once** into registers `w9`–`w17`;
- keeps **three pointers** (`x7`, `x8`, `x2`) on the three image rows under the kernel, and
  moves them 4 bytes to the right after each output pixel. No index arithmetic at all;
- writes out the 9 kernel positions one after another (no `ky`, `kx` loops).

`ldr w20, [x7], #4` loads a pixel **and** moves the pointer to the next one, in a single
instruction (*post-increment*). One output pixel of v2:

```asm
2:      ldr     w20, [x7], #4           // row y:   in[y][x], then p0 += 4
        mul     w19, w20, w9            //   acc  = pixel * k[0]
        ldr     w20, [x7]               //          in[y][x+1]
        mul     w20, w20, w10           //   product = pixel * k[1]
        add     w19, w19, w20           //   acc += product
        ...                             //   (the same for the other 7 positions)
        str     w19, [x1], #4           // *out++ = acc
        subs    x6, x6, #1              // columns left--
        b.ne    2b
```

### v2 → v3: the MAC instruction (your turn, Part 5)

In v2 every kernel position after the first needs `mul` and then `add`, exactly as you
would write it in Ripes: RISC-V (RV32IM) has no instruction that does both. ARM64 does: a
**multiply-accumulate** (MAC) instruction:

```asm
madd    Wd, Wn, Wm, Wa      // Wd = Wa + Wn * Wm
```

### v3 → v4: NEON, 4 pixels at once

The Raspberry Pi's CPU has **NEON** vector registers (`v0`–`v31`) of 128 bits, which hold
4 numbers of 32 bits each. One vector instruction works on all 4 at the same time:

```asm
ldr     q16, [x7]                   // load 4 pixels: in[y][x .. x+3]
mla     v20.4s, v16.4s, v0.s[1]     // 4 accumulators += 4 pixels * k[1]
```

v4 computes **4 output pixels per loop iteration**. The C compiler did not use these
instructions for `conv.c`; writing assembly by hand let us use them.

✏️ **Q8.** Count the instructions that run for **one output pixel** in v2 (loads,
multiplies, adds, the store, and the 2 loop instructions). How many is that per
multiply-add? Compare with your answer to Q4 for assembly v1.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 5. Hands-on: write v3 yourself

Open **[src/conv_student.S](src/conv_student.S)** (click the link, or use the Jupyter file
browser). It starts as an **exact copy of v2**, so right now the "ASM v3 MAC (yours)" row is
`OK` but no faster than v2.

**Your task:** the file has 8 places marked `TODO`, each a `mul` + `add` pair:

```asm
        mul     w20, w20, w10           // TODO 1: fuse this MUL ...
        add     w19, w19, w20           //         ... and this ADD into one MADD
```

Replace each pair with **one** `madd` instruction that gives the same result. Read the
comments at the top of the file first: they say which register holds what.

Save the file (Ctrl + S in the editor tab), come back here and run the cell below. It
rebuilds the code and runs the benchmark **without** Python (so it is quick); the reference
is then C v1.

- `NOT BUILT`: the assembler found an error. Read the message above the table, fix the
  line it names, and run again.
- `MISMATCH`: it builds but computes the wrong result. Check the order of the operands.
- To start over: `!git checkout src/conv_student.S`

In [ ]:
!make && python3 conv_bench.py --skip-python

The cell below prints the `madd` lines in your file, as proof of your solution in the
exported notebook. It must show 8 lines.

In [ ]:
!grep -nE "^\s+madd" src/conv_student.S

✏️ **Q9.** Compare the time of your v3 with v2. How many instructions per output pixel
does v3 run (use your count from Q8)? Did the speed change by the same proportion as the
instruction count? Propose one reason why it might not.

**Your answer:** *(double-click this cell and write it here)*

---
## Part 6. The race

Now see it live, on your photo. Python processes it on a **CPU core of its own**. Meanwhile,
on another core, the six C and assembly versions process the same photo **one after
another**, never at the same time, so no version slows another one down.

- **Panel 1 (one photo):** the progress of the photo for every version. The C and assembly
  bars fill one after another in less than a second, each with its time, while Python's
  bar crawls for about 9 seconds.
- **Panel 2:** how many photos each C and assembly version **could** process in the time
  Python has used so far. It is calculated, not run: *Python's time ÷ that version's time
  for one photo*. It grows while Python works. When the longest bar reaches the end, the
  scale grows (*full bar = N photos*) and all bars shrink together.

Run the cell and watch the bars (about 15 seconds including the preparation).

In [ ]:
race = notebook_race()

✏️ **Q10.** How long did Python take for the photo in the race, and how does it compare
with its time in Part 3? According to panel 2, how many photos could each version process
in that time? Is the order of the versions the same as in your table from Part 3?

**Your answer:** *(double-click this cell and write it here)*

---
## Part 7. Conclusions (200 words maximum)

Write your conclusions **in your own words** inside the quotes below, then run the cell to
count the words. Answer these points, using your numbers:

1. Where was the biggest jump in speed, and what causes it?
2. Is writing assembly by hand always faster than C? Use ASM v1 vs C v2 as evidence.
3. What did the pointers (v2), the MAC instruction (v3) and NEON (v4) each add?
4. Why does it matter to understand what code does at the lowest level?

In [ ]:
conclusions = """
Write your conclusions here.
"""

words = len(conclusions.split())
print(f"{words} words:", "OK" if words <= 200 else "too long, the maximum is 200")

---
## Hand in

1. Check that every cell has its output and every ✏️ question has your answer.
2. Save (**Ctrl + S**).
3. Export: **File → Save and Export Notebook As… → HTML** (in the classic interface:
   **File → Download as → HTML**).
4. Upload the HTML file to Moodle.